In [1]:
import numpy as np
import scipy
from Pauli_path_Heis import *
import time



In [8]:
# Define phase for Heisenberg gates
phi = np.pi/30

# Data needed for the simulation
trans_codes, probs, is_commuting, branch_sign, amp_factr = build_transition_tables(phi)


n_qubits = 20

#Build Trotter step gate pairs
GTL = []
for j in range(0, n_qubits, 2):
    GTL.append([j, j+1])
for j in range(1, n_qubits-1, 2):
    GTL.append([j, j+1])


#Inital Pauli 0:I, 1:X, 2:Y, 3:Z must be np.array of dtype=np.int8
P0 = np.array([3,3,0,0,0] + [0]*(n_qubits-5) , dtype=np.int8) # ZIZI^(N-5)


n_samples = int(4e7) # Numbers of sample paths
eta=np.exp(-.005)*np.ones(n_qubits) # define noise veector same length of n_qubits
n_steps=8 # number of Trotter steps
gates = np.array(GTL*n_steps, dtype=np.int64) #total number of heisenberg gates

# Main function to run Pauli Path Sampling
t0 = time.time()
P_out, s_out, amp_out,damp_out = evolve_many_paths_with_1q_noise(
    P0,
    gates,
    trans_codes, probs, is_commuting, branch_sign,
    amp_factr,
    eta,
    n_samples
)
print(time.time()-t0)

20.951581954956055


In [9]:
# Net expaction value : Generally will con converge due to exploding variance
contribs=bond_projector_contribs(P_out, s_out, amp_out, damp_out)
O_mean,O_std = contrib_stats(contribs)
print('Mean: ',O_mean, 'std: ',O_std/np.sqrt(n_samples))

Mean:  -0.5656633459136367 std:  0.0015891381398113005


In [10]:
# Build Logs of +- samples dists

'''
DAMP and QS functions should converge to desired precision as long as enough samples are generated. 

AMP and AMP_DAMP will have giant variances

'''
amp_p, damp_p, amp_m, damp_m = split_logs_pm(P_out, s_out, amp_out, damp_out)

def DAMP(P,s): # s scales noise level
    dist = np.exp(-s*P)
    return np.mean(dist),np.std(dist)/np.sqrt(P.shape[0])
def AMP(P):
    dist = np.exp(P)
    return np.mean(dist),np.std(dist)/np.sqrt(P.shape[0])
def AMP_DAMP(P,M,s):
    dist = np.exp(P - s*M)
    return np.mean(dist),np.std(dist)/np.sqrt(P.shape[0])

def QS(PP,s): # OUr Q+- functions of s
    P = PP[0,:]
    M = PP[1,:]
    num = ADMP(P,M,s)[0]
    d1 = AMP(P)[0]
    d2 = DAMP(M,s)[0]
    return num/(d1)

array([[0.59873054, 0.01691482],
       [0.01691482, 0.00061863]])